# Module 4: Deploy to Production

You've trained the model, optimized it, and exported it. Now let's ship it.

In this module, you'll:

1. Build a **FastAPI inference API** with proper error handling
2. **Containerize** it with Docker (production-grade Dockerfile)
3. **Deploy** to Google Cloud Run
4. Understand what to **monitor** in production

---

In [ ]:
# --- Colab / Environment Setup (run this cell first) ---
import os, subprocess

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    if not os.path.exists("/content/pytorch-production-workshop"):
        subprocess.run(["git", "clone", "https://github.com/arj7192/pytorch-production-workshop.git"], cwd="/content", check=True)
    os.chdir("/content/pytorch-production-workshop/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete — GPU:", os.environ.get("COLAB_GPU", "not detected"))

## 4.1 The Inference API

Let's walk through the FastAPI server at `serve/app.py`.

Key production patterns:
- **Model loaded once at startup** (not per-request)
- **Health endpoint** (`/health`) for load balancers
- **Input validation** with Pydantic
- **Structured error responses**
- **Request timing** middleware

In [ ]:
import sys
sys.path.insert(0, '..')

# Let's look at the serving code
from pathlib import Path

app_code = Path('../serve/app.py').read_text()
print(app_code)

### Anatomy of the API

```
POST /generate
{
    "prompt": "The meaning of life is",
    "max_tokens": 50,
    "temperature": 0.8
}
→
{
    "text": "The meaning of life is ...",
    "tokens_generated": 50,
    "latency_ms": 123.4
}
```

```
GET /health
→
{
    "status": "healthy",
    "model_loaded": true,
    "device": "cpu"
}
```

## 4.2 Run Locally

First, test the server locally before containerizing.

In [ ]:
# The server expects a checkpoint and tokenizer in the serve/ directory.
# If no checkpoint exists yet (fresh Colab session), we'll train one first.
import shutil
import torch

serve_dir = Path('../serve')

# Ensure tokenizer exists
tokenizer_src = Path('../tokenizer.json')
if not tokenizer_src.exists():
    from src.data import prepare_wikitext2
    print("Preparing tokenizer...")
    prepare_wikitext2(vocab_size=8192, seq_len=128, tokenizer_path=str(tokenizer_src))

# Ensure checkpoint exists — train if needed
from src.utils import CheckpointManager, set_seed, get_device
ckpt_manager = CheckpointManager('../checkpoints')
latest = ckpt_manager.latest()

if not latest:
    print("No checkpoint found — training a model (3 epochs with AMP)...")
    from src.model import build_model
    from src.data import prepare_wikitext2, create_dataloaders
    from src.evaluate import evaluate

    device = get_device()
    set_seed(42)
    train_ds, val_ds, _, tok = prepare_wikitext2(vocab_size=8192, seq_len=128, tokenizer_path=str(tokenizer_src))
    cfg = {'vocab_size': tok.get_vocab_size(), 'd_model': 256, 'n_heads': 4, 'd_ff': 512, 'n_layers': 4, 'max_seq_len': 128, 'dropout': 0.1}
    mdl = build_model(cfg).to(device)
    opt = torch.optim.AdamW(mdl.parameters(), lr=3e-4)
    loader, val_loader = create_dataloaders(train_ds, val_ds, batch_size=64)
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda') if use_amp else None

    for epoch in range(3):
        mdl.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp, dtype=torch.float16):
                loss = mdl(x, targets=y)['loss']
            if scaler:
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
                opt.step()
            opt.zero_grad(set_to_none=True)
        val_m = evaluate(mdl, val_loader, device, use_amp=use_amp)
        print(f"  Epoch {epoch+1}/3 | val_loss {val_m['val_loss']:.4f}")
    ckpt_manager.save(mdl, opt, 3, 0, val_m['val_loss'], cfg)
    latest = ckpt_manager.latest()
    print(f"  Checkpoint saved: {latest}")

# Copy artifacts to serve/
shutil.copy(tokenizer_src, serve_dir / 'tokenizer.json')
print(f"Copied tokenizer to {serve_dir / 'tokenizer.json'}")

shutil.copy(latest, serve_dir / 'model_checkpoint.pt')
print(f"Copied checkpoint to {serve_dir / 'model_checkpoint.pt'}")

### Start the server

**Locally** you'd run `cd serve && uvicorn app:app --host 0.0.0.0 --port 8000` in a terminal.

In **Colab** (no terminal), we start it in a background thread and test from the next cell.

In [ ]:
import subprocess, time, threading, requests

# Start uvicorn in a background process
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="../serve",
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)

# Stream server logs in a background thread so we can see startup
def _print_logs(proc):
    for line in proc.stderr:
        print(f"[server] {line.decode().strip()}")
log_thread = threading.Thread(target=_print_logs, args=(server_proc,), daemon=True)
log_thread.start()

# Wait for the server to be ready
print("Starting server...")
for _ in range(30):
    try:
        requests.get("http://localhost:8000/health", timeout=1)
        print("Server is up!\n")
        break
    except requests.ConnectionError:
        time.sleep(1)
else:
    print("Server failed to start — check logs above.")

In [ ]:
# --- Test the API ---

# Health check
resp = requests.get("http://localhost:8000/health")
print("=== Health Check ===")
print(resp.json())

# Generate text
print("\n=== Text Generation ===")
for prompt in ["The future of AI", "In the beginning", "Scientists recently"]:
    resp = requests.post(
        "http://localhost:8000/generate",
        json={"prompt": prompt, "max_tokens": 40, "temperature": 0.8},
    )
    result = resp.json()
    print(f"\nPrompt: {result['prompt']}")
    print(f"Output: {result['text'][:150]}")
    print(f"Latency: {result['latency_ms']:.1f} ms | Tokens: {result['tokens_generated']}")

In [ ]:
# Shut down the server
server_proc.terminate()
server_proc.wait()
print("Server stopped.")

## 4.3 Deploying to Google Cloud Run

We've verified the API works locally. Now let's deploy it as a **scalable cloud service**.

**The plan:**
1. **Here in Colab** — set up a GCP project and upload the trained checkpoint to Google Cloud Storage
2. **In Google Cloud Shell** — clone the repo, pull the checkpoint, build a Docker image, and deploy to Cloud Run

Cloud Run gives us: auto-scaling, HTTPS, load balancing, and pay-per-request pricing.

> **Note for attendees:** If you don't have a GCP account or can't set up billing, follow along with the instructor's demo.

In [ ]:
# --- GCP Project Setup (runs in Colab) ---
import subprocess

print("=" * 55)
print("  GCP PROJECT SETUP")
print("=" * 55)

# Step 1: Authenticate
print("\n[Step 1/4] Authenticating with Google Cloud...")
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("           Done.")
else:
    print("           Not in Colab — run 'gcloud auth login' in your terminal.")

# Step 2: List projects and pick one, or create a new one
print("\n[Step 2/4] Selecting GCP project...")
proj_list = subprocess.run(
    ["gcloud", "projects", "list", "--format", "value(projectId)"],
    capture_output=True, text=True
)
projects = [p.strip() for p in proj_list.stdout.strip().splitlines() if p.strip()]

if projects:
    print("           Your projects:")
    for i, p in enumerate(projects, 1):
        print(f"             {i}. {p}")
    PROJECT_ID = input("\n           Enter a project ID from above (or press Enter to create a new one): ").strip()
else:
    print("           No existing projects found.")
    PROJECT_ID = input("           Enter a project ID to create (or press Enter for auto-generated): ").strip()

if not PROJECT_ID and not projects:
    import random, string
    suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
    PROJECT_ID = f"pytorch-workshop-{suffix}"

if not PROJECT_ID:
    raise SystemExit("No project ID entered. Re-run this cell.")

if PROJECT_ID not in projects:
    print(f"           Creating project: {PROJECT_ID} ...")
    result = subprocess.run(
        ["gcloud", "projects", "create", PROJECT_ID, "--name", "PyTorch Workshop"],
        capture_output=True, text=True
    )
    if result.returncode != 0 and "already exists" not in result.stderr:
        print(f"           Error: {result.stderr.strip()}")
    else:
        print(f"           Created.")

subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID],
               capture_output=True, check=True)
print(f"           Active project: {PROJECT_ID}")

# Step 3: Check billing
print("\n[Step 3/4] Checking billing...")
billing = subprocess.run(
    ["gcloud", "billing", "projects", "describe", PROJECT_ID,
     "--format", "value(billingAccountName)"],
    capture_output=True, text=True
)
billing_linked = bool(billing.stdout.strip())

if billing_linked:
    print(f"           Billing active: {billing.stdout.strip()}")
else:
    print("           No billing account linked.")
    print("           Cloud Run requires billing. Link one at:")
    print(f"           https://console.cloud.google.com/billing/linkedaccount?project={PROJECT_ID}")
    print("           Then re-run this cell.")

# Step 4: Enable APIs
print("\n[Step 4/4] Enabling APIs (Cloud Build, Cloud Run, Storage)...")
if not billing_linked:
    print("           Skipped — link billing first, then re-run this cell.")
else:
    result = subprocess.run([
        "gcloud", "services", "enable",
        "cloudbuild.googleapis.com",
        "run.googleapis.com",
        "storage.googleapis.com",
    ], capture_output=True, text=True)
    if result.returncode == 0:
        print("           Done.")
    else:
        print(f"           Error: {result.stderr.strip()}")

apis_ok = billing_linked and (result.returncode == 0 if billing_linked else False)

print("\n" + "=" * 55)
print(f"  Project:  {PROJECT_ID}")
print(f"  Billing:  {'Linked' if billing_linked else 'NOT LINKED'}")
print(f"  APIs:     {'Enabled' if apis_ok else 'Not enabled'}")
print(f"  Status:   {'Ready' if apis_ok else 'Fix above issues, then re-run'}")
print("=" * 55)

# --- Upload trained checkpoint + tokenizer to GCS ---
import subprocess
from pathlib import Path

BUCKET = f"gs://{PROJECT_ID}-workshop-artifacts"

print("=" * 55)
print("  UPLOAD MODEL ARTIFACTS TO GCS")
print("=" * 55)

# Step 1: Create bucket
print(f"\n[Step 1/3] Creating bucket: {BUCKET}")
result = subprocess.run(
    ["gsutil", "mb", "-l", "us-central1", BUCKET],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("           Created.")
elif "already exists" in result.stderr or "409" in result.stderr:
    print("           Already exists — reusing.")
else:
    print(f"           Warning: {result.stderr.strip()}")

# Step 2: Upload tokenizer
print("\n[Step 2/3] Uploading tokenizer.json...")
tok_path = Path("../serve/tokenizer.json")
if not tok_path.exists():
    tok_path = Path("../tokenizer.json")
subprocess.run(["gsutil", "cp", str(tok_path), f"{BUCKET}/tokenizer.json"],
               capture_output=True, check=True)
print("           Done.")

# Step 3: Upload checkpoint
print("\n[Step 3/3] Uploading model_checkpoint.pt...")
ckpt_path = Path("../serve/model_checkpoint.pt")
subprocess.run(["gsutil", "cp", str(ckpt_path), f"{BUCKET}/model_checkpoint.pt"],
               capture_output=True, check=True)
print("           Done.")

# Summary with Cloud Shell commands
print("\n" + "=" * 55)
print("  Artifacts uploaded. In Cloud Shell, run:")
print(f"  gsutil cp {BUCKET}/tokenizer.json serve/")
print(f"  gsutil cp {BUCKET}/model_checkpoint.pt serve/")
print("=" * 55)

# Let's look at the Dockerfile we'll build in Cloud Shell
from pathlib import Path

print("=== serve/Dockerfile ===\n")
print(Path('../serve/Dockerfile').read_text())

In [ ]:
## 4.4 Docker Build + Deploy (Google Cloud Shell)

Docker and `gcloud` can't run inside Colab, so we switch to **Google Cloud Shell** — a free Linux terminal with Docker and `gcloud` pre-installed.

**Open Cloud Shell now:** [shell.cloud.google.com](https://shell.cloud.google.com)

---

### Step 1 — Clone the repo and pull your trained checkpoint

```bash
git clone https://github.com/arj7192/pytorch-production-workshop.git
cd pytorch-production-workshop

gsutil cp gs://$GOOGLE_CLOUD_PROJECT-workshop-artifacts/tokenizer.json serve/
gsutil cp gs://$GOOGLE_CLOUD_PROJECT-workshop-artifacts/model_checkpoint.pt serve/
```

**No checkpoint?** Train one directly in Cloud Shell:

```bash
pip install -r requirements.txt
python -m src.train --config configs/fast_debug.yaml
cp tokenizer.json serve/
cp checkpoints_debug/checkpoint_*.pt serve/model_checkpoint.pt
```

### Step 2 — Build and test Docker locally

```bash
docker build -f serve/Dockerfile -t pytorch-workshop-api .
docker run -d -p 8080:8000 pytorch-workshop-api
```

Test it:

```bash
curl http://localhost:8080/health

curl -X POST http://localhost:8080/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt":"Hello world","max_tokens":30}'
```

### Step 3 — Deploy to Cloud Run

First, build and push the image to Google Container Registry:

```bash
gcloud builds submit \
  --tag gcr.io/$GOOGLE_CLOUD_PROJECT/pytorch-workshop-api \
  -f serve/Dockerfile .
```

Then deploy — **pick CPU or GPU:**

**CPU** (default, no GPU quota needed):

```bash
gcloud run deploy pytorch-workshop-api \
  --image gcr.io/$GOOGLE_CLOUD_PROJECT/pytorch-workshop-api \
  --platform managed --region us-central1 \
  --port 8000 --memory 2Gi --cpu 2 \
  --allow-unauthenticated
```

**GPU** (requires GPU quota):

```bash
gcloud run deploy pytorch-workshop-api \
  --image gcr.io/$GOOGLE_CLOUD_PROJECT/pytorch-workshop-api \
  --platform managed --region us-central1 \
  --port 8000 --memory 4Gi --cpu 4 --gpu 1 --gpu-type nvidia-l4 \
  --allow-unauthenticated
```

> The same Docker image works for both — `serve/app.py` auto-detects GPU at startup and falls back to CPU.

### Step 4 — Test your live endpoint

```bash
SERVICE_URL=$(gcloud run services describe pytorch-workshop-api \
  --region us-central1 --format "value(status.url)")

curl $SERVICE_URL/health

curl -X POST $SERVICE_URL/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt":"The future of AI","max_tokens":40}'
```

Or use the deploy script for a one-liner:

```bash
chmod +x serve/deploy.sh
./serve/deploy.sh $GOOGLE_CLOUD_PROJECT          # CPU
./serve/deploy.sh $GOOGLE_CLOUD_PROJECT --gpu     # GPU
```

## 4.5 What to Monitor in Production (Reference)

Once deployed, you need to know when things go wrong. Key metrics:

| Metric | What it tells you | Alert threshold |
|--------|------------------|----------------|
| **Request latency (P99)** | User experience | >500ms for sync APIs |
| **Error rate** | Model/service health | >1% |
| **Memory usage** | OOM risk | >80% of limit |
| **Cold start time** | Time to first request | >10s |
| **Model staleness** | When was the model last updated | App-specific |

### Cloud Run gives you:
- Request count, latency, error rate (built-in)
- Container CPU/memory usage (built-in)
- Custom metrics via Cloud Logging (add to your app)

### Application-level logging (already in our API):
- Request ID for tracing
- Inference latency per request
- Input/output sizes

## Summary: What We Built Today

```
┌─────────────────────────────────────────────────────────┐
│                  Workshop Pipeline                      │
│                                                         │
│  ┌─────────┐   ┌──────────┐   ┌──────────┐   ┌──────┐ │
│  │ Module 1 │──▶│ Module 2 │──▶│ Module 3 │──▶│Mod. 4│ │
│  │ Train   │   │ Optimize │   │ Export   │   │Deploy│ │
│  └─────────┘   └──────────┘   └──────────┘   └──────┘ │
│                                                         │
│  Transformer     AMP           TorchScript    FastAPI   │
│  Training loop   Profiling     ONNX          Docker    │
│  Checkpoints     Stability     Quantization  Cloud Run │
│  Eval + logging  DataLoader    Benchmarking  Monitoring│
└─────────────────────────────────────────────────────────┘
```

You now have a complete, production-grade ML pipeline — from a raw model to a deployed, monitored inference service.

### Next steps for your own projects:
1. **Swap the model**: Replace `TransformerLM` with your model
2. **Scale up**: Add GPU Cloud Run instances for large models
3. **Add CI/CD**: GitHub Actions for automatic deployment
4. **A/B testing**: Deploy multiple model versions side by side
5. **Monitoring**: Add Prometheus metrics or Cloud Monitoring